# Biohub: Learned U-Net + Transformer + ILP Candidate

**Objective:** run the exact-validated learned/global pipeline on all four test
movies and write a schema-checked `submission.csv`.

Locked validation evidence: exact score `0.8394088969`, which is `+0.0289511808`
above the best classical validation. Parameters are unchanged from that run.


## Locked candidate plan

1. Use the same public artifact, T4 GPU, split-0 checkpoint, and isolated runtime.
2. Run detection threshold `0.99` and ILP division weight `1.0` on every test movie.
3. Convert predicted GEFF graphs to the competition CSV inside the isolated
   environment.
4. Assert dataset coverage, node-ID uniqueness, and valid edge endpoints.
5. Save compact per-dataset statistics and a run manifest.

This notebook writes an output file but never submits it to the competition.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

COMPETITION = "biohub-cell-tracking-during-development"
METHOD = "unet_transformer"
SEED = 42
np.random.seed(SEED)

DET_THRESHOLD = 0.99
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
UNET_BATCH_SIZE = 4
MAX_VALIDATION_EMBRYOS = 3
CLASSICAL_BENCHMARK = 0.8104577160678552

COMP_DIR = next(
    (path for path in [
        Path(f"/kaggle/input/competitions/{COMPETITION}"),
        Path(f"/kaggle/input/{COMPETITION}"),
    ] if path.exists()),
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
)
TRAIN_DIR = COMP_DIR / "train"
TEST_DIR = COMP_DIR / "test"
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING / "cellmot_repo"
LEARNED_SITE = WORKING / "cellmot_learned_site"
LEARNED_MARKER = LEARNED_SITE / ".install_complete"

print("Competition:", COMP_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: this learned validation is intended for a GPU kernel")


## Offline artifact and isolated dependencies

The notebook has internet disabled. All repository code, wheels, and weights come
from the attached public artifact dataset. Dependency wheels are installed under
`cellmot_learned_site` and exposed only to subprocesses through `PYTHONPATH`.


In [ ]:
def find_artifacts() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/thibautgoldsborough/"
             "cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts"),
    ]
    root = Path("/kaggle/input")
    if root.exists():
        for child in root.iterdir():
            if "cellmot" in child.name.lower():
                candidates.extend([child, child / child.name])
    for candidate in candidates:
        if ((candidate / "repo").exists()
                and (candidate / "weights").exists()
                and (candidate / "wheels").exists()):
            return candidate
    raise FileNotFoundError("Attach thibautgoldsborough/cellmot-baseline-artifacts")


ARTIFACTS = find_artifacts()
if not LEARNED_MARKER.exists():
    LEARNED_SITE.mkdir(exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
        "--ignore-installed", "--target", str(LEARNED_SITE),
        "--no-index", "--find-links", str(ARTIFACTS / "wheels"),
        "tracksdata", "zarr>=3.0.10", "pyscipopt",
    ], check=True)
    LEARNED_MARKER.write_text("ok\n")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)

LEARNED_ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join([str(LEARNED_SITE), str(REPO_DIR / "src")]),
    "PYTHONNOUSERSITE": "1",
}

weights_root = REPO_DIR / "weights" / METHOD
available_splits = sorted(path.name for path in weights_root.iterdir() if path.is_dir())
if not available_splits:
    raise FileNotFoundError(f"No learned checkpoints under {weights_root}")
SPLIT_NAME = "split_0" if "split_0" in available_splits else available_splits[0]
WEIGHTS = weights_root / SPLIT_NAME / "edge_predictor_best.pth"
if not WEIGHTS.exists():
    raise FileNotFoundError(WEIGHTS)

environment_check = subprocess.run([
    sys.executable, "-c",
    "import numpy, torch, zarr, tracksdata, pyscipopt; "
    "print(numpy.__version__, torch.__version__, torch.cuda.is_available())",
], env=LEARNED_ENV, capture_output=True, text=True, check=True)

print("Artifacts:", ARTIFACTS)
print("Available splits:", available_splits)
print("Selected checkpoint:", WEIGHTS.relative_to(REPO_DIR))
print("Isolated environment:", environment_check.stdout.strip())


## Isolated GEFF-to-submission writer

The learned repository emits GEFF graph directories. The writer below reads them
with isolated Zarr, verifies graph integrity, and writes the required flat CSV.


In [ ]:
SUBMISSION_WRITER = WORKING / "learned_submission_writer.py"
SUBMISSION_WRITER.write_text(r"""
import csv
import sys
from pathlib import Path

import numpy as np
import zarr

output_path = Path(sys.argv[1])
prediction_paths = [Path(value) for value in sys.argv[2:]]
columns = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]

row_id = 0
with output_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()
    for prediction_path in prediction_paths:
        group = zarr.open(str(prediction_path), mode="r")
        node_ids = np.asarray(group["nodes/ids"][:], dtype=np.int64)
        node_t = np.asarray(group["nodes/props/t/values"][:], dtype=np.int64)
        node_z = np.asarray(group["nodes/props/z/values"][:], dtype=np.float64)
        node_y = np.asarray(group["nodes/props/y/values"][:], dtype=np.float64)
        node_x = np.asarray(group["nodes/props/x/values"][:], dtype=np.float64)
        edges = np.asarray(group["edges/ids"][:], dtype=np.int64).reshape(-1, 2)

        if len(set(int(value) for value in node_ids)) != len(node_ids):
            raise AssertionError(f"Duplicate node IDs in {prediction_path.stem}")
        node_id_set = set(int(value) for value in node_ids)
        if any(int(source) not in node_id_set or int(target) not in node_id_set
               for source, target in edges):
            raise AssertionError(f"Dangling edge endpoint in {prediction_path.stem}")

        dataset = prediction_path.stem
        for index, node_id in enumerate(node_ids):
            writer.writerow({
                "id": row_id, "dataset": dataset, "row_type": "node",
                "node_id": int(node_id), "t": int(node_t[index]),
                "z": int(round(float(node_z[index]))),
                "y": int(round(float(node_y[index]))),
                "x": int(round(float(node_x[index]))),
                "source_id": -1, "target_id": -1,
            })
            row_id += 1
        for source_id, target_id in edges:
            writer.writerow({
                "id": row_id, "dataset": dataset, "row_type": "edge",
                "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": int(source_id), "target_id": int(target_id),
            })
            row_id += 1
""")
print("Isolated GEFF submission writer ready")


## Test datasets

The candidate must produce exactly one graph for every test movie. Dataset names
are derived from the attached competition input, never hard-coded.


In [ ]:
def list_zarr_names(directory: Path) -> list[str]:
    return sorted(path.stem for path in directory.glob("*.zarr"))


test_samples = list_zarr_names(TEST_DIR)
if not test_samples:
    raise RuntimeError("No test movies found")
print(f"Test movies ({len(test_samples)}):", test_samples)


## Locked learned prediction

This uses the exact-validated default parameters. Stdout and stderr are retained
as output artifacts for auditability.


In [ ]:
def run_learned_prediction(data_dir: Path, sample_names: list[str]) -> tuple[list[Path], float]:
    splits_file = REPO_DIR / "splits_locked_test.json"
    splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": sample_names}]))

    prediction_root = REPO_DIR / "predictions"
    if prediction_root.exists():
        shutil.rmtree(prediction_root)

    command = [
        sys.executable, "scripts/predict_unet_transformer.py",
        "--data-dir", str(data_dir),
        "--splits", splits_file.name,
        "--split", "0",
        "--weights", str(WEIGHTS.relative_to(REPO_DIR)),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
        "--use-ilp",
    ]
    started = time.time()
    result = subprocess.run(
        command, cwd=REPO_DIR, env=LEARNED_ENV,
        capture_output=True, text=True,
    )
    runtime_seconds = time.time() - started
    (WORKING / "learned_predict.stdout.log").write_text(result.stdout)
    (WORKING / "learned_predict.stderr.log").write_text(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Learned prediction failed with code {result.returncode}:\n"
            + result.stderr[-5000:]
        )

    predictions = sorted(prediction_root.rglob("*.geff"))
    prediction_by_name = {path.stem: path for path in predictions}
    missing = sorted(set(sample_names) - set(prediction_by_name))
    if missing:
        raise RuntimeError(f"Missing learned predictions: {missing}")
    return [prediction_by_name[name] for name in sample_names], runtime_seconds


print("Prediction command configured for", SPLIT_NAME)


## Full test inference and schema validation

This is the expensive cell. It runs all test movies once, writes the submission,
and verifies the flat-table contract in the main kernel as a second check.


In [ ]:
prediction_paths, prediction_runtime = run_learned_prediction(TEST_DIR, test_samples)

submission_path = WORKING / "submission.csv"
writer_result = subprocess.run([
    sys.executable, str(SUBMISSION_WRITER), str(submission_path),
    *[str(path) for path in prediction_paths],
], env=LEARNED_ENV, capture_output=True, text=True)
if writer_result.returncode != 0:
    raise RuntimeError("Submission writer failed:\n" + writer_result.stderr[-4000:])

submission = pd.read_csv(submission_path)
expected_columns = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]
assert list(submission.columns) == expected_columns
assert submission["id"].is_unique
assert set(submission["dataset"]) == set(test_samples)
assert not submission.isna().any().any()
assert set(submission["row_type"]).issubset({"node", "edge"})

stats = []
for dataset, group in submission.groupby("dataset"):
    nodes = group[group.row_type == "node"]
    edges = group[group.row_type == "edge"]
    node_ids = set(int(value) for value in nodes.node_id)
    assert nodes.node_id.is_unique
    assert edges.source_id.isin(node_ids).all()
    assert edges.target_id.isin(node_ids).all()
    out_degree = edges.groupby("source_id").size() if len(edges) else pd.Series(dtype=int)
    stats.append({
        "dataset": dataset,
        "frames": int(nodes.t.nunique()),
        "nodes": int(len(nodes)),
        "edges": int(len(edges)),
        "divisions": int((out_degree >= 2).sum()),
        "nodes_per_frame": round(len(nodes) / max(nodes.t.nunique(), 1), 2),
    })

stats_df = pd.DataFrame(stats).sort_values("dataset")
stats_df.to_csv(WORKING / "learned_test_stats.csv", index=False)
manifest = {
    "method": METHOD,
    "split": SPLIT_NAME,
    "checkpoint": str(WEIGHTS.relative_to(REPO_DIR)),
    "det_threshold": DET_THRESHOLD,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "prediction_runtime_seconds": round(prediction_runtime, 2),
    "submission_rows": int(len(submission)),
    "submission_nodes": int((submission.row_type == "node").sum()),
    "submission_edges": int((submission.row_type == "edge").sum()),
}
(WORKING / "learned_candidate_manifest.json").write_text(json.dumps(manifest, indent=2))

display(stats_df)
print("Candidate manifest:", json.dumps(manifest, indent=2))
print("Wrote:", submission_path)


## Decision record

Inspect `learned_test_stats.csv`, `learned_candidate_manifest.json`, prediction
logs, and `submission.csv`. The output is ready for manual submission only after
the kernel completes and all assertions pass. No automatic competition submission
occurs in this notebook.
